![QuantStudio系统](./images/QuantStudio系统.png)

# QuantStudio 对象

所有的 QuantStudio 对象均继承自 `__QS_Object__`, QuantStudio 对象创建的 `__init__` 方大多会接收三个输入参数：
* args: dict, 默认值 {}, 指定的对象参数集.
* config_file: None 或者文件路径, 默认值 None, 配置文件路径, 配置文件用于设置对象参数. 配置文件是一个 json 格式的文件(字符编码为 utf-8, 扩展名为 json), 以键值对的形式给出各个参数的取值. 
* logger: None 或者日志对象, 默认值 None, 用于内部打印日志.

参数设置的优先级: args > config_file > 内部默认值.

QuantStudio 对象配置文件的默认存放位置为用户目录下的 “QuantStudioConfig” 文件夹, 比如 Windows 系统下通常为 “C:\Users\你的用户名\QuantStudioConfig”, Linux 系统下通常为 “/home/你的用户名/QuantStudioConfig”, Mac OS 下通常为 “/Users/你的用户名/QuantStudioConfig”, 或者可以运行下面的代码获取该路径:

In [1]:
from QuantStudio import __QS_ConfigPath__
print(__QS_ConfigPath__)

C:\Users\admin\QuantStudioConfig


QuantStudio 对象有三个基本属性：
* QSID: str, 表示对象行为的全局唯一 id，且每次运行程序时该 id 不变。相同 QSID 的对象行为一致，但不同的 QuantStudio 对象有可能 QSID 相等。
* Args: QSArgs, 该对象的参数集对象
* Logger: 日志对象

QuantStudio 对象有一个基本方法：`new(self, args={}, **kwargs)`，给定新的参数集 args 产生一个新的 QuantStudio 对象

In [4]:
# QuantStudio 对象基本属性
import logging
from QuantStudio.Factor.HDF5DB import HDF5DB

FDB = HDF5DB(args={"MainDir": "./data/HDF5"}, config_file="./config/HDF5DBConfig.json", logger=logging.getLogger("QS"))
print("QSID : ", FDB.QSID)
print("Args : ", FDB.Args)
print("Logger : ", FDB.Logger)
print("=============================")
NewFDB = FDB.new(args={"Name": "NewHDF5DB"})
print("QSID : ", NewFDB.QSID)
print("Args : ", NewFDB.Args)
print("Logger : ", NewFDB.Logger)

2026-03-02 11:22:59,345 | QS | WARNING : 找不到配置文件: C:\Users\admin\QuantStudioConfig\./config/HDF5DBConfig.json


QSID :  c10190f1688a6555a5fb5095e203e79f7d541993cfb7e210a22bfd7dea986e8a
Args :  HDF5DB.QSArgs(Name='HDF5DB', MainDir=WindowsPath('data/HDF5'), LockDir=None, FileOpenRetryNum=inf, ProcessLock=True)
Logger :  <Logger QS (INFO)>
QSID :  ef6f6d9ee9ca2e8139ef94dff63b40278f88d0547f9cd89efa3acc866138f0c6
Args :  HDF5DB.QSArgs(Name='NewHDF5DB', MainDir=WindowsPath('data/HDF5'), LockDir=None, FileOpenRetryNum=inf, ProcessLock=True)
Logger :  <Logger QS (INFO)>


# 参数集对象
所有的 QuantStudio 对象均有一个参数集对象的属性 Args，参数集类似于一个 dict，某些参数决定了 QuantStudio 对象的行为。

所有的参数集对象均继承自 `QSArgs`, 参数集有三个基本属性：
* QSID: str, 表示对象行为的全局唯一 id，且每次运行程序时该 id 不变。相同 QSID 的对象行为一致，但不同的 QuantStudio 对象有可能 QSID 相等。
* Logger: 日志对象
* Owner: 参数集所属的 QuantStudio 对象，如果为 None 表示该参数集不属于任何 QuantStudio 对象

参数集对象实现了如下方法：
* `__iter__`: 用于迭代参数
* `__getitem__`: 给定参数名称, 获取参数值
* `__setitem__`: 设置参数

In [5]:
# 参数获取和修改
print("修改前: ")
for iArgName, iArgVal in FDB.Args:
    print(iArgName, " : ", iArgVal)
print("=============================")

FDB.Args["FileOpenRetryNum"] = 3

print("修改后: ")
for iArgName, iArgVal in FDB.Args:
    print(iArgName, " : ", iArgVal)

修改前: 
Owner  :  <QuantStudio.Factor.HDF5DB.HDF5DB object at 0x000001FB7F7833B0>
Logger  :  <Logger QS (INFO)>
Name  :  HDF5DB
MainDir  :  data\HDF5
LockDir  :  None
FileOpenRetryNum  :  inf
ProcessLock  :  True
修改后: 
Owner  :  <QuantStudio.Factor.HDF5DB.HDF5DB object at 0x000001FB7F7833B0>
Logger  :  <Logger QS (INFO)>
Name  :  HDF5DB
MainDir  :  data\HDF5
LockDir  :  None
FileOpenRetryNum  :  3
ProcessLock  :  True


参数集对象实现了如下方法：
* `to_dict(self, repr=True)`: 以 dict 形式返回所有参数和参数值
    + repr: bool, 是否仅返回可见参数
* `meta(self, key: Optional[Literal["title", "description", "frozen", "exclude", "repr"]]=None, repr=True)`: 给定 key，返回参数集中参数的元信息，
    + key：元信息的 key, 可选值有
        - title：str, 参数的说明性名称
        - description：str, 参数的描述信息
        - frozen：bool, 参数值是否可以改变, 默认值 False
        - exclude：bool, False 表示用于生成参数集的 QSID，即该参数会影响对象的行为, 默认值 False
        - repr: bool, 该参数是否可见, 默认值 True
    + repr: bool, 是否仅返回可见参数
* `info(self, repr=True, html=False)`: 返回参数的说明信息，主要包括 title 和 description
    + repr: bool, 是否仅返回可见参数
    + html: bool, 是否返回 html 格式的说明信息，False 返回 markdown 格式

In [6]:
# 参数信息
from IPython.display import HTML
print(FDB.Args.to_dict(repr=False))
print("=============================")
print(FDB.Args.meta(key="frozen"))
print("=============================")
display(HTML(FDB.Args.info(html=True)))

{'Owner': <QuantStudio.Factor.HDF5DB.HDF5DB object at 0x000001FB7F7833B0>, 'Logger': <Logger QS (INFO)>, 'Name': 'HDF5DB', 'MainDir': WindowsPath('data/HDF5'), 'LockDir': None, 'FileOpenRetryNum': 3, 'ProcessLock': True}
Name                 True
MainDir              True
LockDir              True
FileOpenRetryNum    False
ProcessLock          True
dtype: bool


# 计算图框架

QuantStudio 中的计算以有向图的形式表达，图由若干个节点组成，每个节点完成某种定义的计算，节点之间的依赖关系以有向边来表示，边从依赖节点指向被依赖的节点。

每个计算节点必须继承自 `QuantStudio.Core.Node.Node`，对象创建的 `__init__` 方法除了 QuantStudio 对象的三个输入参数外还有一个 deps 参数，其为当前节点依赖的节点列表。

要实现一个计算节点，必须实现 Node 的五个方法: init_compute, prepare_compute, forward_compute, backward_compute, merge_result, 其中除 backward_compute 外均有默认实现，backward_compute 是节点运算的主逻辑实现。
```
class Node(__QS_Object__):
    """节点类，以节点为中心的计算单元"""

    def __init__(self, deps:List["Node"]=[], args:dict={}, config_file:Optional[str]=None, **kwargs):
        pass
    
    def init_compute(self, path: List[str], init_data: Any, context: Context) -> List[Any]:
        """
        按照边的方向传递数据执行初始化，可以修改 context 中的全局变量，最好不要有耗时的计算
        :param path: 运行至当前节点的路径, 所有上游节点 ID 的 list
        :param init_data: 上游传递的数据
        :param context: 全局上下文对象
        :return: 产生的向下游传递的数据列表, 如果返回空 list 表示终止继续向下的初始化
        """
        if self.QSID in path: return []
        return [init_data] * len(self.Deps)

    def prepare_compute(self, prepare_data: Any, context: Context) -> None:
        """
        准备计算，不可以修改 context 中的全局变量，最好将 IO 操作在这里实现，只对 context.PrepareNodeDict 中的节点执行该操作
        :param context: 全局上下文对象
        :return: 无返回值
        """
        pass

    def forward_compute(self, path: List[str], fwd_data: Any, context: Context) -> Tuple[List[Any], Any]:
        """
        按照边的方向传递数据执行运算
        :param path: 运行至当前节点的路径, 所有上游节点 ID 的 list
        :param fwd_data: 上游传递的数据
        :param context: 运算时全局上下文对象
        :return: (产生的向下游传递的数据列表, 局部运行时上下文), 如果返回空数据列表表示终止继续向下的运算, 局部运行时上下文将传递给 backward_compute 方法作为入参
        """
        return [fwd_data] * len(self.Deps), fwd_data

    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context: Any=None) -> Any:
        """
        按照边的反方向传递数据执行运算
        :param path: 运行至当前节点的路径, 所有上游节点 ID 的 list
        :param bwd_data_list: 下游传递的数据列表, 如果为空列表表示在 forward_compute 方法中选择了终止向下的运算
        :param context: 运算时全局上下文对象
        :param local_context: 运算时局部上下文对象
        :return: 产生的消息列表
        """
        raise NotImplementedError("子类必须实现 backward_compute 方法")
    
    def merge_result(self, result_list: List[Any], context: Context) -> Any:
        """
        合并并行计算产生的结果
        :param result_list: 并行计算产生的结果列表
        :param context: 运算时全局上下文对象
        :return: 合并后的结果
        """        
        return result_list
```

节点计算方法的入参里有一个全局上下文对象 Context，其继承自 QSArgs，主要用于维护节点的状态等信息，定义如下：
```
class Context(QSArgs):
    NodeDict: Dict[str, "Node"] = Field(default={}, title="节点集", description="{节点ID: Node}, 本次运算的所有 Node, 由计算引擎生成")
    NodeState: Dict[str, Any] = Field(default={}, title="节点状态", description="{节点ID: Any}, 运算中用于存储节点的临时数据，由节点生成和维护")
    PrepareNodeDict: Dict[str, Tuple[str, Any]] = Field(default={}, title="准备节点列表", description="{准备ID: (节点ID, Any)}, 需要执行准备操作的节点列表")
    PID: str = Field(default="0", title="当前进程ID", description="当前的运行进程 ID, 默认为 '0'")
    PIDList: List[str] = Field(default=["0"], title="全部进程ID", description="所有运行进程 ID 列表")
    SplitType: Literal["连续切分", "间隔切分"] = Field(default="连续切分", title="切分方式", frozen=True)
    Event: dict = Field(default={}, title="", description="{节点ID: (Sub2MainQueue, Event)}, 用于多进程同步的 Event 数据")
```

计算图的调度和运行由计算引擎对象 Engine 执行，每种计算引擎的实现均继承自 `QuantStudio.Core.CalcEngine.Engine`，其执行计算的主要方法是
```
def run(self, node_list: List[Node], context: Context, init_data_list: Optional[List[Any]]=None, fwd_data_list: Optional[List[Any]]=None) -> List[Any]:
    """
    给定节点列表, 执行所有节点的计算, 返回每个节点的计算结果
    :param node_list: 节点列表
    :param context: 全局上下文对象
    :param init_data_list: 初始化数据列表
    :param fwd_data_list: 前向计算输入数据列表
    :return: 节点计算结果列表
    """
```

In [ ]:
# 使用计算图框架实现四则运算
from typing import Any, List, Optional

import numpy as np
import pandas as pd

from QuantStudio.Core.Node import Node, Context
from QuantStudio.Core.CalcEngine import Engine

class Num(Node):
    """数字"""
    def __init__(self, value:float, deps:List["Node"]=[], args:dict={}, config_file:Optional[str]=None, **kwargs):
        if "Name" not in args: args = args | {"Name": str(value)}
        self._Value = value
        return super().__init__(deps=deps, args=args, config_file=config_file, **kwargs)

    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return self._Value

class Add(Node):
    """加法"""
    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return np.sum(bwd_data_list)

class Minus(Node):
    """减法"""
    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return bwd_data_list[0] - bwd_data_list[1]

class Prod(Node):
    """乘法"""
    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return np.prod(bwd_data_list)

class Div(Node):
    """除法"""
    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return bwd_data_list[0] / bwd_data_list[1]


Node1 = Prod([Add([Num(1), Num(2)]), Num(3)], args={"Name": "(1 + 2) * 3"})
Node2 = Add([Num(3), Prod([Num(3), Num(2)])], args={"Name": "3 + 3 * 2"})

Engine = Engine()
NodeList = [Node1, Node2]
Rslt = Engine.run(NodeList, Context())
for i, iNode in enumerate(NodeList):
    print(iNode.Name, "=", Rslt[i])

(1 + 2) * 3 = 9
3 + 3 * 2 = 9


In [8]:
from mermaid import Mermaid
from QuantStudio.Tools.Visualization import node2dict, dict2mermaid

NodeDict = node2dict([Node1, Node2])
NodeMermaid = dict2mermaid(NodeDict)
Mermaid(NodeMermaid)